# Setup for using GN for GINNs

### Imports and setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import GeneralNet # make sure the residuals are defined as model attributes
from training.optimizers import GaussNewtonNew
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.visualization.utils_mesh import get_mesh
from training.residuals import bind_model, r_data, r_eikonal, r_principle_curvature_1, r_principle_curvature_2

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### Main training loop

In [2]:
pts_eikonal = torch.zeros(1000, 3, dtype=torch.float64)
pts_boundary = torch.zeros(1000, 3, dtype=torch.float64)
pts_surface = torch.zeros(1000, 3, dtype=torch.float64)
# include connectedness and design region in data. 
# If their weighting should be different this needs to be adapted inside the optimizer code
pts_data = torch.zeros(1000, 3, dtype=torch.float64)
vals_data = torch.zeros(pts_data.shape[0], dtype=torch.float64)

loss_weights = {"data": 1.0, "eikonal": 0.001, "surface_strain": 1.0}

config = {
    "pts_data": pts_data,
    "vals_data": vals_data,
    "pts_eikonal": pts_eikonal,
    "pts_surface": pts_surface,
    "loss_weights": loss_weights,
    "regularization": 1e-6,
}

model = GeneralNet(ks=[3, 128, 128, 128, 128, 1])
model = model.double()
torch.compile(model)
bind_model(model)
params = model.params

In [4]:
# I have added line search to the optimizer for better stability
optimizer = GaussNewtonNew(model, lr=1e-1, config=config, do_woodbury=True)

for i in (pbar:=trange(1000)):
    optimizer.zero_grad()
    
    with torch.no_grad():
        # These loss calculations are just for logging, a more efficient way would be to calculate them from the residual
        # inside the optimizer class and output from there
        loss_data  = 0.5*r_data(params, pts_data, vals_data).squeeze(1).square().mean() # This includes the interface loss
        loss_eikonal  = 0.5*r_eikonal(params, pts_eikonal).squeeze(1).square().mean()

        # Sample new surface points, you can replace this with the surface sampling from GINN
        # pts_surface = sample_model_surface_binsearch(model, pts_boundary, bound_limit=2)
        # # Optional: refine surface samples with Newton
        # pts_surface = sample_model_surface_newton(model, pts_surface)
        pts_surface = torch.zeros(1000, 3, dtype=torch.float64)
        # Update the config
        config["pts_surface"] = pts_surface
        optim.config = config
        
        loss_surface_strain = 0.5*(r_principle_curvature_1(params, pts_surface).squeeze(1).square().mean()
                                   + r_principle_curvature_2(params, pts_surface).squeeze(1).square().mean())
           
        loss = loss_weights["data"] * loss_data + loss_weights["eikonal"] * loss_eikonal + loss_weights["surface_strain"] * loss_surface_strain

        pbar.set_description(f"data: {loss_data.item():.2e} "
                            f"eikonal: {loss_eikonal.item():.2e} "
                            f"surface strain: {loss_surface_strain.item():.2e} "
                            f"{len(pts_surface)}"
                            )
    optimizer.step()

  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -1.25]),
    bbox_max=torch.tensor([2, 2, 1.25]),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig.display()
model.double()

Output()

GeneralNet(
  (fcs): ModuleList(
    (0): Linear(in_features=3, out_features=128, bias=True)
    (1-3): 3 x Linear(in_features=128, out_features=128, bias=True)
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)